In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from lib.probability_of_default import hosmer_lemeshow_test, binomial_calibration_test
from lib.model_selection import compare_models, coefficient_checks
from lib.data_quality import data_quality_report, apply_data_quality_rules
from lib.feature_selection import classification_woe_iv
from lib.probability_of_default import ProbabilityOfDefault
from lib.feature_selection import WoETransformer
from lib.feature_engineering import select_columns, log_loan_amount

# Load Data and check it's quality

In [ ]:
df = pd.read_csv("data/credit_risk_dataset.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
df.describe(include="all")

In [ ]:
data_quality_report(df)

The data has 165 duplicated rows, which will cause overoptimistic test results after splitting the data, and has some data quality issues (impossible/unlikely employment length, and high age).

The quality rules below set outliers to null, as they are MNAR and the model can account for this, and removes duplicated rows.

In [ ]:
df.drop_duplicates(inplace=True)
df = apply_data_quality_rules(df)
data_quality_report(df)

Split the data into train and test sets for evaluation. Each split set has it's index reset so they can interact with numpy arrays later more cleanly (particularly the calibration evaluation).

In [ ]:
target_col = "loan_status"

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

X_train = X_train.reset_index(drop=True).copy()
X_test = X_test.reset_index(drop=True).copy()
y_train = y_train.reset_index(drop=True).copy()
y_test = y_test.reset_index(drop=True).copy()

In [ ]:
print(
    "      |   X   | y\n"
    "––––––––––––––––––––––––\n"
    f"Train | {len(X_train)} | {len(y_train)}\n"
    f"Test  | {len(X_test)}  | {len(y_test)}\n"
)

In [ ]:
fig, ax = plt.subplots()

(y_train.value_counts(normalize=True)*100).plot(kind="bar", title="Default distribution in training set", ax=ax)

ax.set_xlabel("Loan Status (Default=1)")
ax.set_ylabel("Percent (%)")

plt.show()

# Feature selection and engineering

In [ ]:
iv, woe = classification_woe_iv(pd.concat([X_train, y_train], axis=1), target_col, show_woe=False)
woe

The below features are selected because they have an information value (IV) that suggests that they are good predictors of default. Note that loan_grade (IV=0.87) and loan_int_rate (IV=0.66) are not included as features in the modelling. This is because these features are circular and also are part of seperate modelling within the same process as PD.

In [ ]:
core_feats = ["person_home_ownership", "person_emp_length", "loan_intent", "cb_person_default_on_file"]

In [ ]:
def build_pipeline(feats, custom_bins=None, class_weight=None):
    return Pipeline([
        ("select", FunctionTransformer(select_columns, kw_args={"columns": feats})),
        ("woe",    WoETransformer(variables=feats, custom_bins=custom_bins)),
        ("model",  LogisticRegression(class_weight=class_weight, random_state=42)),
    ])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0)

**Experiment 1:** How does dropping colinear features affect the model?

In [ ]:
affordability_options = {
    "all three":                ["loan_amnt", "person_income", "loan_percent_income"],
    "drop loan_amnt":           ["person_income", "loan_percent_income"],
    "drop person_income":       ["loan_amnt", "loan_percent_income"],
    "drop loan_percent_income": ["loan_amnt", "person_income"],
}
models = {name: build_pipeline(core_feats + extra_feats) for name, extra_feats in affordability_options.items()}

compare_models(models, X_train, y_train, cv, baseline="drop loan_amnt")

In [ ]:
pd.DataFrame({
    name: {"All signs OK": checks["Sign OK"].all(), "Max VIF": checks["VIF"].max()}
    for name, checks in ((n, coefficient_checks(m, X_train, y_train)) for n, m in models.items())
}).T

Keeping all three affordability variables has the best Gini, but it leads to the coefficient not making sense because there is multicolinearity between them (you can't keep two constant and move the third).

**Experiment 2:** Does coarse classing and WoE transformation of loan amount affect the model?

In [ ]:
fine = WoETransformer(variables=["loan_amnt"]).fit(X_train, y_train)
fine.woe_tables_["loan_amnt"][["Cutoff", "N", "Bads", "WoE"]]

In [ ]:
# Compare the candidate groupings
affordability_feats = ["person_income", "loan_percent_income"]
coarse_candidates = {
    "[5000, 12000]":         [5000, 12000],
    "[4400, 10000, 14500]":  [4400, 10000, 14500],
    "[10000, 15000]":        [10000, 15000],
    "[12000]":               [12000],
}

models = {"drop loan_amnt":  build_pipeline(core_feats + affordability_feats),
          "loan_amnt fine":  build_pipeline(core_feats + affordability_feats + ["loan_amnt"])}
for name, cuts in coarse_candidates.items():
    models[f"loan_amnt {name}"] = build_pipeline(core_feats + affordability_feats + ["loan_amnt"],
                                                 custom_bins={"loan_amnt": cuts})

compare_models(models, X_train, y_train, cv, baseline="drop loan_amnt")

In [ ]:
# View the coarse WoE values for each candidate
for name, cuts in coarse_candidates.items():
    t = WoETransformer(variables=["loan_amnt"], custom_bins={"loan_amnt": cuts}).fit(X_train, y_train)
    print(name, t.woe_tables_["loan_amnt"][["Cutoff", "N", "WoE"]].round(3).values.tolist())

In [ ]:
# Stability check: is the pattern real or noise?
cuts = [4400, 10000, 14500]
fold_woe = []
for i, (tr, _) in enumerate(StratifiedKFold(5, shuffle=True, random_state=1).split(X_train, y_train)):
    t = WoETransformer(variables=["loan_amnt"], custom_bins={"loan_amnt": cuts})
    t.fit(X_train.iloc[tr], y_train.iloc[tr])
    fold_woe.append(t.woe_tables_["loan_amnt"].set_index("Cutoff")["WoE"].rename(f"fold {i+1}"))

pd.concat(fold_woe, axis=1)

- The fine binned WoE transformed loan amount has the best Gini, but it's WoE values are not linear - there is a hump at bins 4/5.
- The coarse bins [5000, 12000] and [4400, 10000, 14500] are the next best Gini models, but their WoE values are also non-linear.
- The next best bins are **[10000, 15000]**. This bin does have linear WoE values, and it makes business sense.

In [ ]:
feats = core_feats + affordability_feats + ["loan_amnt"]
custom_bins = {"loan_amnt": [10000, 15000]}

# Model selection

**Experiment 3:** Check whether class weight balancing and calibration affect the model

In [ ]:
calibration_candidates = {
    "balanced, no calibration":   build_pipeline(feats, custom_bins, class_weight="balanced"),
    "balanced + isotonic":        CalibratedClassifierCV(build_pipeline(feats, custom_bins, "balanced"), method="isotonic", cv=5),
    "unweighted, no calibration": build_pipeline(feats, custom_bins),
    "unweighted + isotonic":      CalibratedClassifierCV(build_pipeline(feats, custom_bins), method="isotonic", cv=5),
    "unweighted + sigmoid":       CalibratedClassifierCV(build_pipeline(feats, custom_bins), method="sigmoid", cv=5),
}

oof_cv = StratifiedKFold(5, shuffle=True, random_state=0)
rows, oof = [], {}
for name, model in calibration_candidates.items():
    p = cross_val_predict(model, X_train, y_train, cv=oof_cv, method="predict_proba")[:, 1]
    oof[name] = p
    hl, _, hl_p = hosmer_lemeshow_test(y_train.values, p)
    low = p <= np.quantile(p, 0.1)
    rows.append({"Model": name,
                 "Gini": 2 * roc_auc_score(y_train, p) - 1,
                 "Mean PD": p.mean(),
                 "Actual DR": y_train.mean(),
                 "Brier": brier_score_loss(y_train, p),
                 "Log loss": log_loss(y_train, p),
                 "HL": hl,
                 "HL p": hl_p,
                 "Low decile pred (%)": round(p[low].mean()*100, 1),
                 "Low decile obs (%)": round(y_train[low].mean()*100, 1)})

pd.DataFrame(rows).set_index("Model").round(4)

In [ ]:
binomial_calibration_test(y_train.values, oof["unweighted + isotonic"], strategy="quantile")

In [ ]:
b = build_pipeline(feats, custom_bins, "balanced").fit(X_train, y_train)[-1]
u = build_pipeline(feats, custom_bins).fit(X_train, y_train)[-1]
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Intercept shift: {b.intercept_[0] - u.intercept_[0]:.3f}  vs  ln(ratio) = {np.log(ratio):.3f}")

Balanced + no calibration has the best Gini 0.6370, but unweighted + isotonic calibration wins in every other field (Gini=0.6319, Brier=0.1186, Log-loss=0.3869, HL_p=0.5465). The binomial goodness-of-fit test for the unweighted + isotonic calibration model also looks good, particularly in the lowest-risk bin (although slight under-prediction).

Final model

In [ ]:
pipe = build_pipeline(feats, custom_bins=custom_bins) # no balencing class weights
pod = ProbabilityOfDefault(pipe, X_train, X_test, y_train, y_test)

# Probability of Default Modelling and Evaluation

## Cross-validation

In [ ]:
pod.cross_validation_metrics()

In [ ]:
pod.run_cv_calibration_tests()

## Model discrimination on test set

In [ ]:
print(f"ROC AUC: {pod.roc_auc():.2f}")

In [ ]:
print(f"PR AUC:   {pod.pr_auc():.2f}")

In [ ]:
print(f"Average precision:   {pod.average_precision():.2f}")

In [ ]:
ginis = []
for seed in range(10):
    a, b, c, d = train_test_split(X, y, stratify=y, test_size=0.2, random_state=seed)
    ginis.append(ProbabilityOfDefault(pipe, a, b, c, d).gini_coefficient())

print(f"Test Gini: {np.mean(ginis):.3f} ± {np.std(ginis):.3f}")

In [ ]:
print(f"Gini coefficient:   {pod.gini_coefficient():.2f}")

In [ ]:
print(f"KS Statistic: {pod.ks_statistic():.2f}")

In [ ]:
print(pod.generate_ranking_stability_report())

In [ ]:
print(f"Brier score:   {pod.brier_score():.2f}")

## Model calibration on test set

In [ ]:
hl_stat, dof, p_value = pod.hosmer_lemeshow_test()
hl_df = pd.DataFrame({
    "Hosmer-Lemeshow Statistic": [hl_stat],
    "Degrees of Freedom": [dof],
    "p-value": [p_value]
})
hl_df

In [ ]:
print(pod.binomial_goodness_of_fit_test(strategy='quantile'))

In [ ]:
fig, ax = plt.subplots()

ax.plot(np.linspace(0, 1, 100), np.linspace(0, 1, 100), color="black", linestyle="--")
pod.binomial_goodness_of_fit_test().plot(x="expected_prob", y="observed_prob", title="LR Calibration", ax=ax)

plt.show()

## Population Stability Index

In [ ]:
psi_report = pod.generate_psi_report(n_bins=10)
print(psi_report.to_string(index=False))

In [ ]:
pod.fit()
test_pds = pod.calibrated_model.predict_proba(X_test)[:, 1]
test_scores = pod.convert_pd_to_credit_score(test_pds)

# Quick verification display mapping PD to Scores
score_samples = pd.DataFrame({
    'Actual Label': y_test,
    'Calibrated PD (%)': np.round(test_pds * 100, 1),
    'Credit Bureau Point Score': test_scores
})
print("\n=== Credit Scoring Samples (Test Set) ===")
print(score_samples.head(10))